# Phase 3: Counterfactual GCM + CDI Precomputation

Fits a DoWhy StructuralCausalModel (GCM) on the SCM training data, then precomputes an item-level Causal Diversity Impact (CDI) cache for the RL agent's reward shaping.

## Scope Lock
- Phase 3 only: GCM fitting and CDI precomputation.
- Uses `src.counterfactual.gcm_fit.fit_gcm()` and `src.counterfactual.precompute_cdi.precompute_cdi_cache()`.
- CDI is computed for a subset of (user, candidate) pairs and cached to `artifacts/cdi_cache.pkl`.

## Outputs
- `artifacts/gcm_model.pkl` — fitted DoWhy StructuralCausalModel
- `artifacts/cdi_cache.pkl` — dict mapping `(user_id, item_id) -> CDI score`

In [ ]:
import warnings
import logging
import pickle
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd

_cwd = Path.cwd()
_root = _cwd.parent if _cwd.name == "notebooks" else _cwd
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from src.counterfactual.gcm_fit import fit_gcm
from src.counterfactual.precompute_cdi import precompute_cdi_cache

warnings.filterwarnings('ignore')
logging.getLogger('dowhy').setLevel(logging.WARNING)
logging.getLogger('src.counterfactual').setLevel(logging.INFO)

DATA = _root / "data"
ARTIFACTS = _root / "artifacts"
ARTIFACTS.mkdir(parents=True, exist_ok=True)
print(f"Project root: {_root}")
print(f"Artifacts: {ARTIFACTS}")

## 2. Load Phase 1 Data
Load `scm_train.parquet` produced by the Phase 1 pipeline.

In [ ]:
data_path = DATA / "scm_train.parquet"
try:
    df = pd.read_parquet(data_path)
    print(f"Loaded Phase 1 data: {len(df)} rows, {df['user_id'].nunique()} users, {df['impression_id'].nunique()} impressions")
except FileNotFoundError:
    print(f"ERROR: {data_path} not found. Please run Phase 1 data pipeline first.")
    raise


## 3. Fit StructuralCausalModel (GCM)
Build the causal DAG from PCA columns and fit a DoWhy GCM with auto-assigned causal mechanisms.

In [ ]:
pca_cols = [c for c in df.columns if c.startswith("U_pca_")]
print(f"Found {len(pca_cols)} PCA columns")

sample = df.sample(n=min(50000, len(df)), random_state=42)
print(f"Fitting GCM on {len(sample)} rows ...")

t0 = time.time()
gcm_model = fit_gcm(sample, pca_cols)
t_gcm = time.time() - t0
print(f"GCM fitted in {t_gcm:.1f} s")

gcm_path = ARTIFACTS / "gcm_model.pkl"
with open(gcm_path, "wb") as f:
    pickle.dump(gcm_model, f)
print(f"GCM saved to {gcm_path}")


## 4. Build Sessions for CDI Precomputation
Create session objects from the SCM data, one per impression. Each session holds the user's history embedding, candidate items, and click labels.

In [ ]:
import ast


def _to_array(val):
    if isinstance(val, (list, np.ndarray)):
        return np.array(val, dtype=np.float32)
    if isinstance(val, str):
        return np.array(ast.literal_eval(val), dtype=np.float32)
    raise TypeError(f"Embedding has unexpected type {type(val)}")


sessions = []
for imp_id, group in df.groupby("impression_id", sort=False):
    user_id = group["user_id"].iloc[0]
    history_emb = _to_array(group["U_history_emb_full"].iloc[0])
    item_ids = group["item_id"].tolist()
    title_embs = group["I_title_emb_full"].apply(_to_array).tolist()
    candidates = [
        type("C", (), {"item_id": iid, "title_emb": emb})()
        for iid, emb in zip(item_ids, title_embs)
    ]
    session = type("Session", (), {
        "user_id": user_id,
        "initial_history_emb": history_emb,
        "candidate_pool": item_ids,
    })()
    sessions.append(session)

print(f"Built {len(sessions)} sessions")


## 5. Precompute CDI Cache
For each (user, candidate) pair, query the GCM to compute the counterfactual Y_diversity under intervention (A=1, with the candidate's category and sentiment), keeping observed confounders fixed. Cache results for the RL agent's reward shaping.

In [ ]:
entity_cols = [c for c in df.columns if c.startswith("I_entity_pca_")]
title_cols = [c for c in df.columns if c.startswith("I_title_pca_")]
select_cols = ["item_id", "I_category", "I_sentiment"] + entity_cols + title_cols
news_df = df[select_cols].drop_duplicates("item_id")
news_df = news_df.set_index("item_id")

categories = sorted(df["I_category"].unique().tolist())
print(f"Categories ({len(categories)}): {categories}")

subset_sessions = sessions[:min(100, len(sessions))]
for sess in subset_sessions:
    sess.candidate_pool = sess.candidate_pool[:5]

t0 = time.time()
cdi_cache = precompute_cdi_cache(
    gcm_model,
    subset_sessions,
    news_df,
    df,
    categories=categories,
    cache_path=ARTIFACTS / "cdi_cache.pkl",
)
t_cdi = time.time() - t0

sample_keys = list(cdi_cache.keys())[:3]
for k in sample_keys:
    print(f"  CDI {k[0]} -> {k[1]}: {cdi_cache[k]:.4f}")
print(f"CDI precomputed in {t_cdi:.1f} s — {len(cdi_cache)} entries")

## 6. Summary
Phase 3 complete. Artifacts generated:

In [ ]:
print(f"GCM model:     {gcm_path}")
print(f"CDI cache:     {ARTIFACTS / 'cdi_cache.pkl'}")
print(f"CDI entries:   {len(cdi_cache)}")
print("\nPhase 3 complete.")